<a href="https://colab.research.google.com/github/f20250495-del/Keyword-spotting-system/blob/main/Kwd_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import tarfile
import urllib.request
import glob
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader

# --- Data Download and Extraction ---
DATA_DIR = "data"
# Note: Sticking to V0.01 as present in the user's initial cell for consistency.
# If V0.02 is preferred, the URL below would need to be updated.
URL = "http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz"
ARCHIVE_PATH = os.path.join(DATA_DIR, "speech_commands.tar.gz")

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
    print(f"Created {DATA_DIR} directory.")

    print("Downloading Google Speech Commands V1 (approx. 1.5GB compressed)...")
    urllib.request.urlretrieve(URL, ARCHIVE_PATH)
    print("Download complete.")

    print("Extracting audio files...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(path=DATA_DIR)

    os.remove(ARCHIVE_PATH)
    print(f"Dataset ready and extracted inside the './{DATA_DIR}' folder!")

# --- Definition of get_data_splits ---
def get_data_splits(data_dir):
    """
    Parses the Google Speech Commands directory structure and the predefined
    validation/testing text files to create clean, leak-free splits.

    Args:
        data_dir (str): Path to the local 'data/' folder containing extracted files.

    Returns:
        train_list (list), val_list (list), test_list (list), labels (list)
    """
    val_txt_path = os.path.join(data_dir, "validation_list.txt")
    test_txt_path = os.path.join(data_dir, "testing_list.txt")

    with open(val_txt_path, "r") as f:
        val_list = [line.strip().replace("\\", "/") for line in f.readlines() if line.strip()]

    with open(test_txt_path, "r") as f:
        test_list = [line.strip().replace("\\", "/") for line in f.readlines() if line.strip()]

    val_set = set(val_list)
    test_set = set(test_list)

    all_wav_paths = glob.glob(os.path.join(data_dir, "*", "*.wav"))

    train_list = []
    unique_labels = set()

    for full_path in all_wav_paths:
        rel_path = os.path.relpath(full_path, data_dir).replace("\\", "/")
        label = os.path.dirname(rel_path)

        if label.startswith("_") or label == ".pytest_cache":
            continue

        unique_labels.add(label)

        if rel_path not in val_set and rel_path not in test_set:
            train_list.append(rel_path)

    sorted_labels = sorted(list(unique_labels))

    total_files = len(train_list) + len(val_list) + len(test_list)
    print("--- Data Splitting Sanity Check ---")
    print(f"Total Audio Samples Found: {total_files}")
    print(f"Train Set:      {len(train_list)} files ({len(train_list)/total_files*100:.1f}%)")
    print(f"Validation Set: {len(val_list)} files ({len(val_list)/total_files*100:.1f}%)")
    print(f"Testing Set:    {len(test_list)} files ({len(test_list)/total_files*100:.1f}%)")
    print(f"Total Unique Classes: {len(sorted_labels)}")

    return train_list, val_list, test_list, sorted_labels

# --- Definition of SpeechDataset ---
class SpeechDataset(Dataset):
    def __init__(self, base_dir):
        self.base_dir = base_dir

        self.sample_rate = 16000
        self.n_fft = 512
        self.hop_length = 320
        self.n_mels = 32

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=self.n_fft, hop_length=self.hop_length, n_mels=self.n_mels
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

        self.file_paths = []
        self.labels = []

        self.class_to_idx = {"yes": 0, "no": 1, "stop": 2, "go": 3, "unknown": 4}

        for category in os.listdir(base_dir):
            category_path = os.path.join(base_dir, category)
            if os.path.isdir(category_path):
                label_idx = self.class_to_idx.get(category, self.class_to_idx["unknown"])

                for filename in os.listdir(category_path):
                    if filename.endswith('.wav'):
                        full_path = os.path.join(category_path, filename)
                        self.file_paths.append(full_path)
                        self.labels.append(label_idx)

    def pad_waveform(self, waveform, target_length=16000):
        if waveform.shape[1] < target_length:
            waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]
        return waveform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        relative_path = self.file_paths[idx]
        current_label = self.labels[idx]

        waveform, sr = torchaudio.load(relative_path)

        waveform = self.pad_waveform(waveform)
        mel_spec = self.mel_transform(waveform)
        mel_spec_db = self.db_transform(mel_spec)

        return mel_spec_db, current_label

# --- Definition of CustomKWSNet ---
import torch.nn.functional as F

class CustomKWSNet(nn.Module):
    def __init__(self, num_classes=5):
        super(CustomKWSNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=(2, 3), stride=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=(2, 2), stride=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc = nn.Linear(16 * 3 * 5, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Quick verification test
# model = CustomKWSNet()
# test_input = torch.randn(1, 1, 32, 51)
# output = model(test_input)

# print(f"Input shape:  {test_input.shape}")
# print(f"Output shape: {output.shape}")
# print(f"Total Params: {sum(p.numel() for p in model.parameters()):,}")


import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

def train_kws_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 15

    # Corrected DATA_DIR to use the local path
    DATA_DIR = "./data"

    # 3. Load Dataset & Create Train/Val Split (80% Train, 20% Val)
    full_dataset = SpeechDataset(base_dir=DATA_DIR)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    print(f"Dataset split: {train_size} training samples | {val_size} validation samples")

    model = CustomKWSNet(num_classes=5).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / train_size
        epoch_train_acc = (correct_train / total_train) * 100

        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = (correct_val / total_val) * 100

        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
              f"| Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.2f}% "
              f"| Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.2f}%")

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), "best_kws_model.pth")
            print(f"  --> Saved new best checkpoint! (Val Acc: {best_val_acc:.2f}%)")

    print("\nTraining completed successfully!")

if __name__ == "__main__":
    train_kws_model()


Created data directory.
Download complete.
Extracting audio files...


/tmp/ipykernel_1225/380286564.py:27: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATA_DIR)


Dataset ready and extracted inside the './data' folder!
Using device: cuda
Dataset split: 51781 training samples | 12946 validation samples
Epoch [01/15] | Train Loss: 0.4352 - Train Acc: 87.46% | Val Loss: 0.3336 - Val Acc: 89.46%
  --> Saved new best checkpoint! (Val Acc: 89.46%)
Epoch [02/15] | Train Loss: 0.2925 - Train Acc: 90.62% | Val Loss: 0.2790 - Val Acc: 91.12%
  --> Saved new best checkpoint! (Val Acc: 91.12%)
Epoch [03/15] | Train Loss: 0.2575 - Train Acc: 91.65% | Val Loss: 0.2665 - Val Acc: 91.47%
  --> Saved new best checkpoint! (Val Acc: 91.47%)
Epoch [04/15] | Train Loss: 0.2431 - Train Acc: 92.12% | Val Loss: 0.2527 - Val Acc: 91.88%
  --> Saved new best checkpoint! (Val Acc: 91.88%)
Epoch [05/15] | Train Loss: 0.2324 - Train Acc: 92.53% | Val Loss: 0.2392 - Val Acc: 92.28%
  --> Saved new best checkpoint! (Val Acc: 92.28%)
Epoch [06/15] | Train Loss: 0.2245 - Train Acc: 92.72% | Val Loss: 0.2297 - Val Acc: 92.82%
  --> Saved new best checkpoint! (Val Acc: 92.82%)
Ep